In [ ]:
import os
import numpy as np
import json
from openai import OpenAI
import fitz
from tqdm import tqdm
from dotenv import load_dotenv

# 简单RAG中的上下文片段标题（CCH）

上下文片段标题（CCH）通过在嵌入每个片段之前为其添加高级上下文（如文档标题或章节标题）来增强RAG。这提高了检索质量并防止了脱离上下文的回复。

## 本笔记本中的步骤：

1. **数据处理**：加载和预处理文本数据。
2. **带上下文标题的分块**：提取章节标题并将其前置到片段中。
3. **嵌入创建**：将带有上下文增强的片段转换为数值表示。
4. **语义搜索**：根据用户查询检索相关片段。
5. **回复生成**：使用语言模型从检索到的文本生成回复。
6. **评估**：使用评分系统评估回复准确性。

In [ ]:

load_dotenv()  # 加载.env文件
api_key = os.getenv("OPENAI_API_KEY")  # 读取密钥
# 初始化 OpenAI 客户端，设置基础 URL 和 API 密钥  
client = OpenAI(  
    base_url="https://api.openai.com/v1/",  
    api_key=os.getenv("OPENAI_API_KEY")  # 从环境变量中获取 API 密钥  
)

## 利用大模型提取段落意思

In [ ]:
def generate_chunk_header(chunk, model="gpt-4o"):
    """
    使用LLM为给定的文本块生成标题/头部。

    参数:
    chunk (str): 作为标题总结的文本块。
    model (str): 用于生成标题的模型。默认值为"gpt-4o"。

    返回:
    str: 生成的标题/头部。
    """
    # 定义系统提示以指导AI的行为
    system_prompt = "从文本中概括出小标题"
    
    # 基于系统提示和文本块从AI模型生成回复
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": chunk}
        ]
    )

    # 返回生成的标题/头部，并去除任何前导/尾随空白字符
    return response.choices[0].message.content.strip()

In [ ]:
def chunk_text_with_headers(text, n, overlap):
    """
    将文本分割为较小的片段并生成标题。

    参数:
    text (str): 要被分割的完整文本。
    n (int): 每个片段的字符长度。
    overlap (int): 片段之间的重叠字符数量。

    返回:
    List[dict]: 包含 'header' 和 'text' 键的字典列表。
    """
    chunks = []  # 初始化一个空列表以存储片段

    # 使用指定的片段大小和重叠值遍历文本
    for i in range(0, len(text), n - overlap):
        chunk = text[i:i + n]  # 提取一个文本片段
        header = generate_chunk_header(chunk)  # 使用LLM生成片段的标题
        chunks.append({"header": header, "text": chunk})  # 将标题和片段追加到列表中

    return chunks  # 返回包含标题和片段的列表

## 向量化

In [ ]:
def create_embeddings(text, model="text-embedding-ada-002"):
    """
    为给定的文本创建嵌入。

    参数:
    text (str): 要嵌入的输入文本。
    model (str): 要使用的嵌入模型。默认值为 "text-embedding-ada-002"。

    返回:
    dict: 包含输入文本嵌入结果的回复。
    """
    # 使用指定的模型和输入文本创建嵌入
    response = client.embeddings.create(
        model=model,
        input=text
    )
    # 从回复中返回嵌入结果
    return response.data[0].embedding

In [ ]:
# 为每个片段生成嵌入
embeddings = []  # 初始化一个空列表以存储嵌入

# 使用进度条迭代每个文本片段
for chunk in tqdm(text_chunks, desc="Generating embeddings"):
    # 为片段的文本创建嵌入
    text_embedding = create_embeddings(chunk["text"])
    # 为片段的标题创建嵌入
    header_embedding = create_embeddings(chunk["header"])
    # 将片段的标题、文本及其嵌入追加到列表中
    embeddings.append({"header": chunk["header"], "text": chunk["text"], "embedding": text_embedding, "header_embedding": header_embedding})

In [ ]:
def cosine_similarity(vec1, vec2):
    """
    计算两个向量之间的余弦相似度。

    参数:
    vec1 (np.ndarray): 第一个向量。
    vec2 (np.ndarray): 第二个向量。

    返回:
    float: 余弦相似度得分。
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

## 标题检索

In [ ]:
def semantic_search(query, chunks, k=5):
    """
    根据查询搜索最相关的文本块。

    参数:
    query (str): 用户查询。
    chunks (List[dict]): 包含嵌入的文本块列表。
    k (int): 返回的结果数量。

    返回:
    List[dict]: 最相关的前k个文本块。
    """
    # 为查询创建嵌入
    query_embedding = create_embeddings(query)

    similarities = []  # 初始化一个列表用于存储相似度分数
    
    # 遍历每个文本块以计算相似度分数
    for chunk in chunks:
        # 计算查询嵌入与文本块文本嵌入之间的余弦相似度
        sim_text = cosine_similarity(np.array(query_embedding), np.array(chunk["embedding"]))
        # 计算查询嵌入与文本块标题嵌入之间的余弦相似度
        sim_header = cosine_similarity(np.array(query_embedding), np.array(chunk["header_embedding"]))
        # 计算平均相似度分数
        avg_similarity = (sim_text + sim_header) / 2
        # 将文本块及其平均相似度分数追加到列表中
        similarities.append((chunk, avg_similarity))

    # 按相似度分数降序排序文本块
    similarities.sort(key=lambda x: x[1], reverse=True)
    # 返回最相关的前k个文本块
    return [x[0] for x in similarities[:k]]